# Quantum Counting

Quantum counting is an algorithm that allows us to estimate how many “marked” elements are in a search space.

If there are:
- $N = 2^n$ possible states ($n$ qubits),
- $M$ of them are marked,

then quantum counting provides an estimate $\tilde M$ with a quadratic speedup compared to classical sampling.

Thus:

- Amplitude amplification is used when $M$ is known or approximately known, we choose the number of iterations $k$ and amplify the probability of finding the marked states.
- Quantum counting: if $M$ is unknown, we first use QPE on $Q$ to obtain $\tilde M$, and then we can optimally choose $k$ in order to implement Amplitude Amplification / Grover search.


## How is it related to Amplitude Amplification

Grover search and Amplitude Amplification act as a rotation in a 2D subspace.

The initial equal superposition is:
$$
|s\rangle = \frac{1}{\sqrt{N}}\sum_{x=0}^{N-1} |x\rangle.
$$

The oracle marks the marked states (phase flips):
$$
S_f|x\rangle = (-1)^{f(x)}|x\rangle.
$$

One iteration of the Grover operator is:
$$
Q = -A S_0 A^\dagger S_f,
$$
where often $A = H^{\otimes n}$, and $|s\rangle = A|0\rangle$.

If $M$ is the number of marked states, then:
$$
\sin^2(\theta) = \frac{M}{N}.
$$

The main idea is that the operator $Q$ acts as a rotation by an angle $2\theta$ in the corresponding marked/unmarked plane.

Since the eigenvalues of any rotation in a two-dimensional plane are complex exponentials, here they are also
$$
e^{\pm i 2\theta}.
$$

From this it follows that, if we are able to determine this phase $2\theta$, then we can calculate $\theta$ and therefore recover the desired quantity $M$.


Quantum counting is essentially Phase Estimation (QPE) applied to the Grover operator $Q$.

Using QPE, it is possible to estimate the phase $\varphi$, where:
$$
e^{2\pi i \varphi} = e^{i2\theta}
\quad\Rightarrow\quad
\varphi = \frac{\theta}{\pi}
\quad (\text{mod } 1).
$$

From the QPE result, we obtain $\tilde\theta$, and then calculate:
$$
\tilde M = N\sin^2(\tilde\theta).
$$

In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFT, GroverOperator
import numpy as np

# phase flip for a specific 3-bit state
def phase_flip(qc, qubits, bitstring):
    q0, q1, q2 = qubits
    bits = [int(b) for b in bitstring]  # converts the text into an array of bit values

    # Transforms the target state into |111> (where bit = 0, applies X)
    if bits[0] == 0: qc.x(q0)
    if bits[1] == 0: qc.x(q1)
    if bits[2] == 0: qc.x(q2)

    qc.h(q2)
    qc.ccx(q0, q1, q2)
    qc.h(q2)

    if bits[0] == 0: qc.x(q0)
    if bits[1] == 0: qc.x(q1)
    if bits[2] == 0: qc.x(q2)


# The oracle marks two states with phase -1
oracle = QuantumCircuit(3, name="Oracle")
phase_flip(oracle, [0, 1, 2], "101")
phase_flip(oracle, [0, 1, 2], "110")

# Creates the Grover operator Q from the oracle
Q = GroverOperator(oracle).to_gate(label="Q")

t = 6         # counting qubits (for precision)
n = 3         # search qubits
N = 2 ** n

qc = QuantumCircuit(t + n, t)

counting = list(range(t))
search = list(range(t, t + n))

# Both registers in superposition
qc.h(counting)
qc.h(search)

# Controlled-Q^(2^k)
cQ = Q.control(1)  # converts the Grover operator into a controlled one
for k in range(t):
    reps = 2 ** k
    for rep in range(reps):
        qc.append(cQ, [counting[k]] + search)  # counting[k] is the control qubit, search are the target qubits

# Inverse QFT on the counting register
qc.append(QFT(t).inverse().to_gate(label="QFT†"), counting)

qc.measure(counting, list(range(t)))

print(qc.draw(fold=120))


sim = AerSimulator()
tqc = transpile(qc, sim)
result = sim.run(tqc, shots=4000).result()
counts = result.get_counts()

print("Counting measurements:", counts)

# Take the most frequent bit string as the phase estimate
top = max(counts, key=counts.get)
y = int(top, 2)
phi = y / (2 ** t)                       # phi ~ theta/pi
M_est = N * (np.sin(np.pi * phi) ** 2)   # M ≈ N sin^2(pi*phi)

print(f"Top = {top}, phi ≈ {phi:.4f}, M_est ≈ {M_est:.2f} (true M=2, N=8)")

C:\Users\ovase\AppData\Local\Temp\ipykernel_11104\3686545189.py:31: DeprecationWarning: The class ``qiskit.circuit.library.grover_operator.GroverOperator`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use qiskit.circuit.library.grover_operator instead.
  Q = GroverOperator(oracle).to_gate(label="Q")
C:\Users\ovase\AppData\Local\Temp\ipykernel_11104\3686545189.py:54: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(t).inverse().to_gate(label="QFT†"), counting)


     ┌───┐                                                                                                            »
q_0: ┤ H ├──■─────────────────────────────────────────────────────────────────────────────────────────────────────────»
     ├───┤  │                                                                                                         »
q_1: ┤ H ├──┼─────■─────■─────────────────────────────────────────────────────────────────────────────────────────────»
     ├───┤  │     │     │                                                                                             »
q_2: ┤ H ├──┼─────┼─────┼─────■─────■─────■─────■─────────────────────────────────────────────────────────────────────»
     ├───┤  │     │     │     │     │     │     │                                                                     »
q_3: ┤ H ├──┼─────┼─────┼─────┼─────┼─────┼─────┼─────■─────■─────■─────■─────■─────■─────■─────■─────────────────────»
     ├───┤  │     │     │     │     │   